# Computational Theory Assessment

## Introduction

This notebook explores the implementation of the SHA-256 cryptographic hash function using Python.

SHA-256 is part of the Secure Hash Algorithm family defined by the National Institute of Standards and Technology (NIST) in the Secure Hash Standard (FIPS 180-4). It produces a fixed-length 256-bit message digest from an input message of arbitrary length.

The objective of this assessment is to investigate the internal operations of SHA-256 and develop an implementation based on the algorithm described in the official standard.

Python will be used as the primary programming language, with its standard library providing functionality for handling binary data, integer conversions and other supporting operations. NumPy will be used where fixed-width unsigned 32-bit integer representations and numerical operations are required.

The implementation is divided into six problems, covering data representation, bitwise operations, constant generation, message padding, the compression function and the complete SHA-256 algorithm.

The completed implementation will be tested against Python's `hashlib` library, which provides an independent reference implementation for verifying the correctness of the generated message digests.

# Problem 1 – Representing SHA-256 Data

Overview

SHA-256 processes information using fixed-size binary data structures. Understanding how these structures are represented in Python is an important first step before implementing the algorithm.

This section investigates the representation of 32-bit words, sequences of words, input messages, 512-bit message blocks, the final hash value and big-endian integers.

For each representation, an appropriate Python data type will be selected, with its suitability demonstrated through practical examples.

### 1.1 Representing 32-bit Words

A word in SHA-256 is a sequence of 32 bits. An unsigned 32-bit integer can represent values ranging from 0 to \(2^{32}-1\).

Python's standard integer type supports arbitrary-precision integers, meaning values are not automatically restricted to 32 bits. NumPy provides the `uint32` data type, which represents unsigned 32-bit integers.

Since SHA-256 performs calculations using fixed-width 32-bit words, NumPy's `uint32` provides a suitable representation for these values.

In [4]:
import numpy as np

# Create an unsigned 32-bit word.
word = np.uint32(0x12345678)

# Display its properties.
print(f"Value: {word}")
print(f"Hexadecimal: {word:08x}")
print(f"Data type: {word.dtype}")
print(f"Size: {word.dtype.itemsize * 8} bits")

Value: 305419896
Hexadecimal: 12345678
Data type: uint32
Size: 32 bits


### 1.2 Representing Input Messages

SHA-256 accepts input messages of arbitrary length. These messages must be converted into bytes before they can be processed by the algorithm.

Python provides the built-in `bytes` type for representing binary data. Text can be converted into bytes using UTF-8 encoding, allowing messages to be represented in a format suitable for cryptographic processing.

The following example demonstrates how a text message is converted into bytes and examines its resulting representation.

[Click here](https://csrc.nist.gov/files/pubs/fips/180-4/final/docs/fips180-4.pdf?)
![UML diagram](img/image.png)

In [5]:
# Define a sample message.
message = "Hello SHA-256"

# Convert the message into bytes.
message_bytes = message.encode("utf-8")

# Display its properties.
print(f"Original message: {message}")
print(f"Byte representation: {message_bytes}")
print(f"Message length: {len(message_bytes)} bytes")
print(f"Data type: {type(message_bytes)}")

Original message: Hello SHA-256
Byte representation: b'Hello SHA-256'
Message length: 13 bytes
Data type: <class 'bytes'>


Once the input message has been converted into bytes, SHA-256 can begin its preprocessing stage. The message is padded and subsequently divided into fixed-size 512-bit blocks before the hashing operations are performed.

### 1.3 Representing 512-bit Message Blocks

SHA-256 processes messages in fixed-size blocks of 512 bits, equivalent to 64 bytes.

Each block consists of sixteen 32-bit words. Messages that do not fit into complete blocks are padded according to the SHA-256 specification before processing.

The following example creates a 64-byte block and examines its size and structure.

In [6]:
# Create a sample 64-byte block.
block = bytes(range(64))

# Display its properties.
print(f"Block size: {len(block)} bytes")
print(f"Block size: {len(block) * 8} bits")
print(f"Number of 32-bit words: {len(block) // 4}")
print(f"Block data: {block.hex()}")

Block size: 64 bytes
Block size: 512 bits
Number of 32-bit words: 16
Block data: 000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f202122232425262728292a2b2c2d2e2f303132333435363738393a3b3c3d3e3f


### 1.4 Converting Bytes into 32-bit Words

Each 512-bit block is divided into sixteen groups of four bytes. Each group is converted into an unsigned 32-bit integer.

![1.4 sha](<img/1.4 sha.jpg>)

SHA-256 uses big-endian byte order when interpreting these groups. This means the most significant byte is placed first.

Python's `int.from_bytes()` method provides a straightforward way to convert each group of bytes into its corresponding integer value.

![1.4py](img/1.4py.jpg)

[Click here](https://docs.python.org/3.12/library/stdtypes.html?utm_source=)


NumPy's `uint32` data type is then used to store the resulting words.
It provides the appropriate fixed-width integer type for storing your SHA-256 words.


In [7]:
# Convert the 64-byte block into sixteen 32-bit words.
words = np.array([
    int.from_bytes(block[i:i+4], byteorder="big")
    for i in range(0, len(block), 4)
], dtype=np.uint32)

# Display the resulting words.
for i, word in enumerate(words):
    print(f"Word {i:2}: 0x{word:08x}")

print(f"\nTotal words: {len(words)}")
print(f"Data type: {words.dtype}")

Word  0: 0x00010203
Word  1: 0x04050607
Word  2: 0x08090a0b
Word  3: 0x0c0d0e0f
Word  4: 0x10111213
Word  5: 0x14151617
Word  6: 0x18191a1b
Word  7: 0x1c1d1e1f
Word  8: 0x20212223
Word  9: 0x24252627
Word 10: 0x28292a2b
Word 11: 0x2c2d2e2f
Word 12: 0x30313233
Word 13: 0x34353637
Word 14: 0x38393a3b
Word 15: 0x3c3d3e3f

Total words: 16
Data type: uint32
